# Mineração de Dados — Data Profiling em 5 Datasets do Scikit-Learn

**Universidade Federal do Ceará | Departamento de Computação**

Disciplina de Mineração de Dados — Prof. José Macedo

---

**Grupo:** _Larissa Vitória, Maria Cecília e Irlisson Ferreira_

---

## Sobre esta entrega

O objetivo desta atividade / notebook é rodar o `ydata-profiling` nos cinco conjuntos de dados do scikit-learn (`iris`, `wine`, `breast_cancer`, `diabetes` e `california housing`) e, a partir do relatório automático, escrever decisões
de análise.

## Parte 0 — Preparação do ambiente

In [1]:
%pip install -q "ydata-profiling[notebook]" scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.3/915.3 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.1 MB/s eta 0:00:00


In [2]:
import sys

import numpy as np
import pandas as pd
import sklearn
import ydata_profiling
from ydata_profiling import ProfileReport

# Registro das versões usadas
# Isso é parte da reprodutibilidade do trabalho
print("python          ", sys.version.split()[0])
print("numpy           ", np.__version__)
print("pandas          ", pd.__version__)
print("scikit-learn    ", sklearn.__version__)
print("ydata-profiling ", ydata_profiling.__version__)

python           3.13.15
numpy            2.1.3
pandas           2.2.3
scikit-learn     1.6.1
ydata-profiling  4.18.4


/tmp/ipykernel_1699/3237502076.py:6: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  import ydata_profiling


## Parte 1 — Carga dos 5 datasets

Todos os cinco Datasets vêm de `sklearn.datasets`.

In [3]:
from sklearn.datasets import (
    load_iris,
    load_wine,
    load_breast_cancer,
    load_diabetes,
    fetch_california_housing,
)

# Um loader por dataset
# Usamos um dicionário para poder iterar sobre os cinco sem repetir código nas próximas partes do notebook!
datasetLoaders = {
    "iris":          load_iris,
    "wine":          load_wine,
    "breast_cancer": load_breast_cancer,
    "diabetes":      load_diabetes,
    "california":    fetch_california_housing,
}

rawBunches = {name: loader(as_frame=True) for name, loader in datasetLoaders.items()}
dataFrames = {name: bunch.frame.copy() for name, bunch in rawBunches.items()}

for name, df in dataFrames.items():
    print(f"{name:<15} {df.shape[0]:>6} linhas x {df.shape[1]:>3} colunas")

iris               150 linhas x   5 colunas
wine               178 linhas x  14 colunas
breast_cancer      569 linhas x  31 colunas
diabetes           442 linhas x  11 colunas
california       20640 linhas x   9 colunas


### Lendo o dicionário de dados

In [4]:
def showDatasetDescription(datasetName: str, charLimit: int = 2000) -> None:
    """Imprime o início do DESCR de um dataset (evita poluir a tela com o texto todo)."""
    print(rawBunches[datasetName].DESCR[:charLimit])

In [5]:
showDatasetDescription("iris")

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 classes.
:Cr

In [7]:
showDatasetDescription("wine")

.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

:Number of Instances: 178
:Number of Attributes: 13 numeric, predictive attributes and the class
:Attribute Information:
    - Alcohol
    - Malic acid
    - Ash
    - Alcalinity of ash
    - Magnesium
    - Total phenols
    - Flavanoids
    - Nonflavanoid phenols
    - Proanthocyanins
    - Color intensity
    - Hue
    - OD280/OD315 of diluted wines
    - Proline
    - class:
        - class_0
        - class_1
        - class_2

:Summary Statistics:

============================= ==== ===== ======= =====
                                Min   Max   Mean     SD
============================= ==== ===== ======= =====
Alcohol:                      11.0  14.8    13.0   0.8
Malic Acid:                   0.74  5.80    2.34  1.12
Ash:                          1.36  3.23    2.36  0.27
Alcalinity of Ash:            10.6  30.0    19.5   3.3
Magnesium:                    70.0 162.0    99.7  14.3

In [10]:
showDatasetDescription("diabetes")

.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi     body mass index
    - bp      average blood pressure
    - s1      tc, total serum cholesterol
    - s2      ldl, low-density lipoproteins
    - s3      hdl, high-density lipoproteins
    - s4      tch, total cholesterol / HDL
    - s5      ltg, possibly log of serum triglycerides level
    - s6      glu, blood sugar level

Note: Each of these 10 feature variables have bee

In [8]:
showDatasetDescription("breast_cancer")

.. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field 20 is Worst Radius.

    - 

In [9]:
showDatasetDescription("california")

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

### Ficha técnica comparativa

Esta tabela é o **item 2 do mini-relatório**. Ela responde de uma vez só ao
passo "Estrutura" e a boa parte do passo "Qualidade" do roteiro, para os
cinco conjuntos lado a lado!!

In [11]:
def buildTechnicalProfile(df: pd.DataFrame) -> dict:
    """Resumo estrutural e de qualidade de um DataFrame: forma, tipos,
    ausentes, duplicatas, colunas constantes e excesso de zeros."""
    rowCount, columnCount = df.shape
    numericColumns = df.select_dtypes(include="number").columns
    constantColumns = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
    totalCells = rowCount * columnCount

    return {
        "linhas": rowCount,
        "colunas": columnCount,
        "numericas": len(numericColumns),
        "nao_numericas": columnCount - len(numericColumns),
        "% ausentes": round(100 * df.isna().sum().sum() / totalCells, 2),
        "duplicatas": int(df.duplicated().sum()),
        "col. constantes": len(constantColumns),
        "% zeros": round(
            100 * (df[numericColumns] == 0).sum().sum() / max(rowCount * len(numericColumns), 1), 2
        ),
        "memoria (KB)": round(df.memory_usage(deep=True).sum() / 1024, 1),
    }


technicalSummary = pd.DataFrame(
    {name: buildTechnicalProfile(df) for name, df in dataFrames.items()}
).T
technicalSummary

,linhas,colunas,numericas,nao_numericas,% ausentes,duplicatas,col. constantes,% zeros,memoria (KB)
iris,150.0,5.0,5.0,0.0,0.0,1.0,0.0,6.67,6.0
wine,178.0,14.0,14.0,0.0,0.0,0.0,0.0,2.37,19.6
breast_cancer,569.0,31.0,31.0,0.0,0.0,0.0,0.0,1.64,137.9
diabetes,442.0,11.0,11.0,0.0,0.0,0.0,0.0,0.00,38.1
california,20640.0,9.0,9.0,0.0,0.0,0.0,0.0,0.00,1451.4


**Pergunta para o relatório:** olhando só para esta tabela, qual dos cinco
conjuntos parece o mais "sujo"? Minha resposta muda depois de abrir os
perfis completos? Por quê?

_(vou responder depois de olhar os cinco relatórios)_

## Parte 2 — Um perfil por dataset

A função abaixo gera e salva o relatório de cada dataset. Os parâmetros
seguem o que vimos em aula (slide 7 de dicas!):

- `minimal=True` para conjuntos com muitas colunas (desliga correlações e
  interações, que ficam caras de calcular);
- amostragem para conjuntos com muitas linhas.

In [12]:
def generateProfileReport(
    datasetName: str,
    df: pd.DataFrame,
    minimal: bool = False,
    sampleSize: int | None = None,
    saveToFile: bool = True,
) -> ProfileReport:
    """Gera o ProfileReport de um dataset e salva como perfil_<nome>.html."""
    dataToProfile = df
    reportTitle = f"Perfil — {datasetName}"

    if sampleSize is not None and len(df) > sampleSize:
        dataToProfile = df.sample(sampleSize, random_state=42)
        reportTitle += f" (amostra de {sampleSize} linhas)"

    profile = ProfileReport(
        dataToProfile, title=reportTitle, explorative=not minimal, minimal=minimal
    )

    if saveToFile:
        outputPath = f"perfil_{datasetName}.html"
        profile.to_file(outputPath)
        print(f"salvo: {outputPath}")

    return profile


# Guardamos cada ProfileReport aqui
profileReports = {}

### 2.1 — Iris
*Linha de base limpa — mas confira as duplicatas.*



In [ ]:
profileReports["iris"] = generateProfileReport("iris", dataFrames["iris"])
profileReports["iris"].to_notebook_iframe()

### Achados — Iris


**1. Existe 1 duplicata exata no dataset**
- Evidência: `df.duplicated().sum() = 1`; linhas 101 e 142 idênticas (sepal length 5.8, sepal width 2.7, petal length 5.1, petal width 1.9), ambas classe *virginica*
- Consequência: só 2 de 150 linhas (0,67%). Portanto, acreditamos que seja uma coincidência de medição, não erro de coleta. Ainda assim, vale remover/marcar antes de um split treino/teste, pra não vazar a mesma amostra pros dois lados!

**2. As 3 classes estão perfeitamente balanceadas**
- Evidência: DESCR — "33.3% for each of 3 classes" :. 50/50/50
- Consequência: não é preciso balanceamento (SMOTE, class_weight); acurácia simples já é uma métrica confiável aqui

**3. Petal length/width são muito mais discriminativas que sepal width**
- Evidência: correlação com a classe no DESCR — petal length 0,9490, petal width 0,9565, sepal width -0,4194
- Consequência: atributos de pétala concentram o poder preditivo; sepal width é candidata a peso baixo (ou até descarte, se fizermos redução de dimensionalidade) num modelo linear


### **Respostas do roteiro (6 passos):**

1. **Contexto:** dataset clássico de Fisher (1936), 150 flores de 3 espécies (Setosa, Versicolor, Virginica), medidas em cm de sépala e pétala, sem ambiguidade de unidade.
2. **Estrutura:** 150 linhas x 5 colunas (4 atributos numéricos + target). O `target` é inferido como numérico (0/1/2), mas semanticamente é categórico (espécie).
3. **Qualidade:** 0% ausentes, 1 duplicata, 0 colunas constantes. Valendo ressaltar que o "6,67% de zeros" da ficha técnica é artefato da própria codificação do target (classe 0 = Setosa conta como zero), não são zeros reais nas medidas físicas.
4. **Distribuições:** conforme a correlação com a classe (petal length 0,9490 / petal width 0,9565), esperamos ver no relatório distribuições bem separadas por espécie nessas duas variáveis — a confirmar assimetria/curtose exatas em `perfil_iris.html`.
5. **Relações:** esperamos correlação forte entre petal length e petal width (ambas muito ligadas à classe) — conferir no relatório se passam de um limiar que indique redundância. Não há risco de vazamento óbvio, já que o alvo é a classe a prever, não um atributo derivado dela.
6. **Decisão:** dataset praticamente pronto pra uso! Tratar a duplicata, dar mais peso às variáveis de pétala, considerar descartar/reduzir peso de sepal width. No mais, como temos classes balanceadas, é relevante o uso de modelos simples (como KNN, ou regressão logística), pois devem performar bem sem pré-processamento pesado.

### 2.2 — Wine
*Escalas muito heterogêneas entre as 13 variáveis.*

In [ ]:
profileReports["wine"] = generateProfileReport("wine", dataFrames["wine"])
profileReports["wine"].to_notebook_iframe()

### Achados — Wine

### **Achados — Wine**

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---------------------|----------------------------------------|----------------------|
| 1 | `proline` e `hue` operam em escalas radicalmente diferentes | `proline`: min 278, max 1680, std 314,9 · `hue`: min 0,48, max 1,71, std 0,23 | A distância euclidiana sem normalização seria dominada por `proline`; padronização torna-se obrigatória antes da aplicação de métodos baseados em distância |
| 2 | `flavanoids` é a variável mais discriminante e também a mais redundante | r=-0,85 com o target; r=0,865 com `total_phenols` | Possui alto poder preditivo isolado, mas usar as duas junto com `total_phenols` acrescenta pouca informação nova, e portanto, torna-se candidata a seleção de features ou PCA |
| 3 | `magnesium` tem cauda pesada e outliers | skew=1,10, curtose=2,10, 4 outliers pelo IQR (limite superior 135,5 vs. máximo real 162) | Justifica transformação log ou modelos robustos a outliers (árvores) em vez de métodos que assumem normalidade |

### **Respostas do roteiro (6 passos):**

1. **Contexto**: dataset que descreve 178 vinhos da mesma região italiana, cultivados por 3 produtores diferentes, caracterizados por 13 medidas químicas contínuas sem unidade comum entre si (de fração adimensional a mg/L), classificadas em 3 classes (59/71/48).
2. **Estrutura**: 178 linhas × 14 colunas (13 features + target); todos os tipos inferidos são numéricos (`float64`/`int64`) e correspondem ao significado semântico, portanto, nenhuma conversão de tipo é necessária.
3. **Qualidade**: 0 ausentes, 0 duplicatas, cardinalidade alta em todas as features (mín. 39 valores únicos em `nonflavanoid_phenols`), sem colunas constantes e sem excesso de zeros nas features.
4. **Distribuições**: `magnesium` (skew 1,10; curtose 2,10) e `malic_acid` (skew 1,04) puxam para a direita com cauda pesada; outliers por IQR concentram-se em `alcalinity_of_ash`, `magnesium` e `color_intensity` (4 cada).
5. **Relações**: `flavanoids` e `total_phenols` são fortemente correlacionadas entre si (r=0,865) e ambas com o target (r=-0,85 e r=-0,72), isso pode ser descrito pela redundância química em que fenóis totais incluem flavonoides, não significa vazamento do rótulo.
6. **Decisão**: a discrepância de escala exige padronização (`StandardScaler`) antes de qualquer método baseado em distância (KNN, k-means, SVM-RBF); a colinearidade entre fenólicos sugere avaliar PCA ou remoção de uma variável redundante para modelos lineares, entretanto, árvores e Naive Bayes toleram ambos os problemas sem pré-processamento.

### 2.3 — Breast Cancer

*Possível redundância: versões mean, error e worst do mesmo atributo.*

In [ ]:
# 30 colunas: começamos pelo modo enxuto (minimal) e depois rodamos sem ele para ver as correlações com calma.
profileReports["breast_cancer"] = generateProfileReport("breast_cancer", dataFrames["breast_cancer"])
profileReports["breast_cancer"].to_notebook_iframe()

In [19]:
# Pista: quantos pares de atributos passam de 0,95 de correlação absoluta?
featureColumns = dataFrames["breast_cancer"].drop(columns="target")
correlationMatrix = featureColumns.corr().abs()

correlationPairs = (
    correlationMatrix.where(np.triu(np.ones(correlationMatrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

print("pares com |r| > 0,95:", int((correlationPairs > 0.95).sum()))
correlationPairs.head(10)

pares com |r| > 0,95: 15


,,0
mean radius,mean perimeter,0.997855
worst radius,worst perimeter,0.993708
mean radius,mean area,0.987357
mean perimeter,mean area,0.986507
worst radius,worst area,0.984015
worst perimeter,worst area,0.977578
radius error,perimeter error,0.972794
mean perimeter,worst perimeter,0.970387
mean radius,worst radius,0.969539
mean perimeter,worst radius,0.969476


### Achados — Breast Cancer

### **Achados — Breast Cancer**

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---|---|---|
| 1 | As tríades radius–perimeter–area (mean, SE e worst) são quase colineares entre si | `mean radius` × `mean perimeter` r=0,998; `mean radius` × `mean area` r=0,987; padrão se repete em SE e worst — 15 pares no total com \|r\|>0,95 | Manter as três versões de cada tríade infla multicolinearidade sem agregar informação; dá pra descartar `perimeter` e `area` e ficar só com `radius` (ou aplicar PCA) por grupo |
| 2 | `concavity`, `compactness` e `concave points` também formam um cluster redundante (r entre 0,83 e 0,92) | correlação par a par: compactness×concavity=0,88; concavity×concave points=0,92; compactness×concave points=0,83 | Mesmo tratamento: reduzir para 1 feature representativa por cluster ou usar regularização L1/L2 em vez de seleção manual |
| 3 | Escalas muito distintas entre variáveis (`mean area` de 143 a 2501 vs `mean smoothness` de 0,05 a 0,16) e assimetria forte em várias colunas `_error` (`area error` skew=5,4; `concavity error` skew=5,1) | describe() / seção de estatísticas do profiling | Modelos sensíveis a escala (regressão logística, SVM, KNN, PCA) exigem `StandardScaler`; a assimetria nas colunas SE sugere log-transform ou uso de modelos baseados em árvore, que são invariantes a isso |

### **Respostas do roteiro (6 passos):**

**1. Contexto**: O dataset descreve 30 atributos numéricos extraídos de imagens digitalizadas de aspirado por agulha fina (FNA) de massas mamárias, descrevendo forma/textura dos núcleos celulares; cada atributo original (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension) aparece em três versões, mean, standard error e worst, totalizando as 30 colunas.

**2. Estrutura**: 569 linhas × 31 colunas (30 features + target). Todos os tipos são `float64`, consistente com o caráter contínuo das medidas, não há coluna categórica mal tipada.

**3. Qualidade**: Zero valores ausentes, zero duplicatas e zero colunas constantes. Classes moderadamente desbalanceadas: 357 benignos vs 212 malignos (~63%/37%), o que já pede atenção em métricas de avaliação (F1, AUC) em vez de accuracy pura.

**4. Distribuições**: Assimetria concentrada nas colunas `_error`: `area error` (skew=5,45) e `concavity error` (skew=5,11) são as mais distorcidas, puxadas por outliers positivos. Pelo critério IQR, `area error` tem 11,4% de outliers e `radius error`/`perimeter error` cerca de 6,7%: os maiores índices do dataset. As versões `mean` e `worst` são bem mais comportadas (skew geralmente <1).

**5. Relações**: 15 pares de atributos ultrapassam \|r\|=0,95, todos dentro das tríades radius/perimeter/area (mean, SE e worst se correlacionando inclusive entre si). Não há vazamento do alvo (não existe feature que seja função direta do label), mas há sinal: `worst concave points` (r=0,79) e `worst perimeter` (r=0,78) são as variáveis mais correlacionadas com o diagnóstico, indicando que tamanho e irregularidade do contorno são os principais marcadores discriminativos.

**6. Decisão**: Antes de modelar:
1. Padronizar todas as features (escalas muito diferentes, ex. area vs smoothness);
2. Reduzir a redundância das tríades radius/perimeter/area e do cluster compactness/concavity/concave points, via remoção manual, PCA ou regularização L1;
3. Dado o desbalanceamento moderado, usar validação estratificada e métricas além de accuracy;
4. Modelos baseados em árvore (Random Forest, Gradient Boosting) toleram bem a multicolinearidade e a assimetria sem pré-processamento extra, sendo um bom baseline antes de investir em engenharia de features para modelos lineares.

### 2.4 — Diabetes

*Já vem centrado e escalonado; `sex` é categórica disfarçada de número.*

In [ ]:
profileReports["diabetes"] = generateProfileReport("diabetes", dataFrames["diabetes"])
profileReports["diabetes"].to_notebook_iframe()

In [20]:
# Pista: os atributos já vêm centrados e escalonados? Quantos valores distintos tem "sex"?
diabetesSummary = dataFrames["diabetes"].describe().T[["mean", "std", "min", "max"]]
diabetesSummary["n_unicos"] = dataFrames["diabetes"].nunique()
diabetesSummary

,mean,std,min,max,n_unicos
age,-2.511817e-19,0.047619,-0.107226,0.110727,58
sex,1.230790e-17,0.047619,-0.044642,0.050680,2
bmi,-2.245564e-16,0.047619,-0.090275,0.170555,163
bp,-4.797570e-17,0.047619,-0.112399,0.132044,100
s1,-1.381499e-17,0.047619,-0.126781,0.153914,141
s2,3.918434e-17,0.047619,-0.115613,0.198788,302
s3,-5.777179e-18,0.047619,-0.102307,0.181179,63
s4,-9.042540e-18,0.047619,-0.076395,0.185234,66
s5,9.293722e-17,0.047619,-0.126097,0.133597,184
s6,1.130318e-17,0.047619,-0.137767,0.135612,56


### Achados — Diabetes

> Três descobertas, cada uma com as **três partes**: observação, evidência numérica e consequência prática.

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---------------------|----------------------------------------|----------------------|
| 1 |                     |                                        |                      |
| 2 |                     |                                        |                      |
| 3 |                     |                                        |                      |

**Respostas do roteiro (6 passos):**

1. Contexto:
2. Estrutura:
3. Qualidade:
4. Distribuições:
5. Relações:
6. Decisão:

### 2.5 — California Housing

*Outliers extremos e alvo truncado no teto da escala.*

In [ ]:
# 20.640 linhas: a amostra dá um perfil rápido; confirmo os outliers no
# conjunto completo antes de escrever qualquer achado.
profileReports["california"] = generateProfileReport("california", dataFrames["california"], sampleSize=5000)
profileReports["california"].to_notebook_iframe()

In [22]:
# Pista: olho os máximos e o topo da escala do alvo.
print(dataFrames["california"][["AveRooms", "AveOccup", "MedHouseVal"]].describe().T)
print()

targetCeiling = dataFrames["california"]["MedHouseVal"].max()
rowsAtCeiling = (dataFrames["california"]["MedHouseVal"] >= targetCeiling).sum()

print(f"valor máximo do alvo: {targetCeiling}")
print(
    f"linhas exatamente no máximo: {rowsAtCeiling} "
    f"({100 * rowsAtCeiling / len(dataFrames['california']):.1f}% do total)"
)

               count      mean        std       min       25%       50%  \
AveRooms     20640.0  5.429000   2.474173  0.846154  4.440716  5.229129   
AveOccup     20640.0  3.070655  10.386050  0.692308  2.429741  2.818116   
MedHouseVal  20640.0  2.068558   1.153956  0.149990  1.196000  1.797000   

                  75%          max  
AveRooms     6.052381   141.909091  
AveOccup     3.282261  1243.333333  
MedHouseVal  2.647250     5.000010  

valor máximo do alvo: 5.00001
linhas exatamente no máximo: 965 (4.7% do total)


### Achados — California Housing

> Três descobertas, cada uma com as **três partes**: observação, evidência numérica e consequência prática.

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---------------------|----------------------------------------|----------------------|
| 1 |                     |                                        |                      |
| 2 |                     |                                        |                      |
| 3 |                     |                                        |                      |

**Respostas do roteiro (6 passos):**

1. Contexto:
2. Estrutura:
3. Qualidade:
4. Distribuições:
5. Relações:
6. Decisão:

---
## Parte 3 — Análise cruzada

Agora, comparamos os cinco entre si!

In [23]:
# Quadro comparativo
targetSummary = {}

for name, df in dataFrames.items():
    targetColumn = df["target"] if "target" in df else df["MedHouseVal"]

    if targetColumn.nunique() <= 20:
        classCounts = targetColumn.value_counts().sort_index()
        distributionDescription = " / ".join(str(v) for v in classCounts.values)
        targetType = f"{targetColumn.nunique()} classes"
        classBalance = round(classCounts.min() / classCounts.max(), 2)
    else:
        distributionDescription = f"{targetColumn.min():.2f} a {targetColumn.max():.2f}"
        targetType = "contínuo"
        classBalance = np.nan

    targetSummary[name] = {
        "tipo do alvo": targetType,
        "distribuição": distributionDescription,
        "balanceamento": classBalance,
    }

pd.concat([technicalSummary, pd.DataFrame(targetSummary).T], axis=1)

,linhas,colunas,numericas,nao_numericas,% ausentes,duplicatas,col. constantes,% zeros,memoria (KB),tipo do alvo,distribuição,balanceamento
iris,150.0,5.0,5.0,0.0,0.0,1.0,0.0,6.67,6.0,3 classes,50 / 50 / 50,1.0
wine,178.0,14.0,14.0,0.0,0.0,0.0,0.0,2.37,19.6,3 classes,59 / 71 / 48,0.68
breast_cancer,569.0,31.0,31.0,0.0,0.0,0.0,0.0,1.64,137.9,2 classes,212 / 357,0.59
diabetes,442.0,11.0,11.0,0.0,0.0,0.0,0.0,0.00,38.1,contínuo,25.00 a 346.00,NaN
california,20640.0,9.0,9.0,0.0,0.0,0.0,0.0,0.00,1451.4,contínuo,0.15 a 5.00,NaN


## Parte 4 — Recomendações de preparação

Para cada dataset, propomos um pipeline curto e **justificamos cada etapa com
um achado do profiling**.

In [24]:
# Esqueleto que vou adaptar para cada dataset. Não preciso treinar modelo
# aqui — o foco desta parte é a preparação dos dados, não a modelagem.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

examplePipeline = Pipeline([
    ("scaler", StandardScaler()),  # justificativa: <achado do profiling>
])
examplePipeline

Pipeline(steps=[('scaler', StandardScaler())])

### Recomendações de preparação por dataset
> VAMOS AJUSTAR / REFINIR ISSO DEPOIS DE TODAS AS ANÁLISES!!!!!!!

**Iris**
- Etapa: remover/marcar a duplicata (linha 101 ou 142) | Achado: 1 duplicata exata (`df.duplicated().sum() = 1`)
- Etapa: `StandardScaler` nos 4 atributos | Achado: mesmo com todos em cm, a escala de pétala difere o bastante da de sépala pra afetar modelos sensíveis a distância (KNN, SVM); é menos crítico aqui que nos outros, mas ainda é boa prática

**Diabetes**
- Etapa: tratar `sex` como categórica (`OneHotEncoder`), não como numérica contínua | Achado: só 2 valores únicos apesar do tipo float
- Etapa: nenhuma escala adicional nos demais 9 atributos | Achado: já vêm centrados e escalonados pelo próprio scikit-learn (`std aprox. 0,0476` em todos, conforme DESCR)

**California Housing**
- Etapa: `RobustScaler` (em vez de `StandardScaler`) em `AveRooms`/`AveOccup` | Achado: outliers extremos (`AveRooms` máx. 141,9; `AveOccup` máx. 1243,3)
- Etapa: flagar/excluir as 965 linhas (4,7%) com `MedHouseVal` no teto (5,00001) antes de treinar uma regressão | Achado: valor truncado/censurado no topo da escala, não uma observação real


**Wine**
- Etapa: `StandardScaler` obrigatório antes de qualquer modelo baseado em distância | Achado: escalas extremamente diferentes entre variáveis (`proline` 278–1680, std≈315 vs `hue` 0,48–1,71, std≈0,23)
- Etapa: Remover uma variável redundante (`flavanoids` ou `total_phenols`) ou aplicar PCA | Achado: correlação de 0,865 entre ambas, além de serem as duas mais correlacionadas com o target (r=-0,85 e r=-0,72)
- Etapa: Divisão treino/teste com estratificação | Achado: classes moderadamente desbalanceadas (class_0=33,1%, class_1=39,9%, class_2=27,0%), não representado no pipeline, pois é aplicado na divisão dos dados, não no pipeline do sklearn.

> Observação de pipeline: Dispensável para árvores e Naive Bayes, pois esses modelos toleram diferença de escala e colinearidade sem pré-processamento

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

wine_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    # justificativa: escalas extremamente diferentes entre variáveis
    # (proline 278–1680, std≈315 vs hue 0,48–1,71, std≈0,23) —
    # sem isso, distância euclidiana seria dominada por proline

    ("pca", PCA(n_components=0.95)),
    # justificativa: flavanoids e total_phenols são colineares (r=0,865);
    # PCA absorve a redundância sem exigir remoção manual de coluna
])

wine_pipeline

Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=0.95))])

**Breast Cancer**
- Etapa: `StandardScaler` seguido de redução de dimensionalidade (PCA) ou seleção de atributos | Achado: 15 pares de variáveis com |r| > 0,95 (ex.: mean radius × mean perimeter = 0,998), forte redundância entre versões mean/SE/worst da mesma medida geométrica
- Etapa: Reduzir cluster compactness/concavity/concave points via remoção manual ou regularização L1/L2 | Achado: correlações entre 0,83 e 0,92 entre as três variáveis
- Etapa: Validação estratificada e métricas além de accuracy (F1, AUC) | Achado: classes moderadamente desbalanceadas (357 benignos vs 212 malignos, ~63%/37%), não representado no pipeline, pois é aplicado na divisão dos dados, não no pipeline do sklearn.
- Etapa: Considerar log-transform ou modelos baseados em árvore | Achado: forte assimetria em colunas `_error` (`area error` skew=5,4; `concavity error` skew=5,1)

> Observação de pipeline: Dispensável para árvores, que toleram diferença de escala e assimetria sem pré-processamento.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

bc_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    # justificativa: escalas muito distintas entre variáveis
    # (mean area 143–2501 vs mean smoothness 0,05–0,16)

    ("pca", PCA(n_components=0.95)),
    # justificativa: 15 pares de features com |r| > 0,95 tornam a
    # matriz de covariância quase singular para modelos lineares;
    # PCA condensa a informação redundante das tríades e do cluster
    # compactness/concavity/concave points em componentes ortogonais
])

bc_pipeline

Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=0.95))])

---
## Parte 6 — Checklist de entrega

Rodamos a célula abaixo antes de fechar o notebook: ela confere se os cinco
HTMLs realmente foram gerados na pasta.

In [25]:
from pathlib import Path

expectedFiles = [f"perfil_{name}.html" for name in datasetLoaders]

for fileName in expectedFiles:
    filePath = Path(fileName)
    if filePath.exists():
        print(f"[ok]    {fileName:<28} {filePath.stat().st_size / 1024:>8.0f} KB")
    else:
        print(f"[FALTA] {fileName}")

[ok]    perfil_iris.html                 1128 KB
[ok]    perfil_wine.html                 7157 KB
[ok]    perfil_breast_cancer.html       74497 KB
[ok]    perfil_diabetes.html             6899 KB
[ok]    perfil_california.html           5244 KB
